<a href="https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/06_tools/notebook_exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6: Tools

This notebook explores **Tools (Function Calling)**, one of the key building blocks of an AI agent. 

We will use the `google-genai` library to interact with Google's Gemini models.

**Learning Objectives:**

1.  **Understand and implement tool use (function calling)** from scratch to allow an LLM to interact with external systems.
2.  **Build a custom tool calling framework** using decorators similar to production frameworks like LangGraph.
3.  **Use Gemini's native tool calling API** for production-ready implementations.
4.  **Implement structured data extraction** using Pydantic models as tools for reliable structured outputs.
5.  **Run tools in a loop** to handle multi-step tasks and understand the limitations that lead to the popular ReAct pattern.

> **Exercise version.** This is the exercise notebook for Lesson 6. The full solution lives in [`notebook.ipynb`](https://colab.research.google.com/github/towardsai/agentic-ai-engineering-course/blob/main/lessons/06_tools/notebook.ipynb) in the same folder. Attempt each exercise before checking the solutions.

### Exercise roadmap

| Exercise | Difficulty | What you build |
|---|---|---|
| 1 | Starter | Hand-written JSON schemas for two tools |
| 2 | Starter | The parser that extracts a tool call from raw LLM text |
| 3 | Advanced | The `@tool` decorator that auto-generates schemas from signatures |
| 4 | Intermediate | Gemini's native tool calling configuration |
| 5 | Starter | A Pydantic model declared as a callable tool |
| 6 | Advanced | The naive multi-step tool calling loop |

## 1. Setup

### Set Up Python Environment

**Google Colab:** Run the code cell below — it installs all required packages and automatically loads your credentials from Colab Secrets (your `GOOGLE_API_KEY`, or your Vertex AI settings if you chose that option in the Course Admin lesson).

To set up your Python virtual environment using `uv` and load it into the Notebook, follow the step-by-step instructions from the `Course Admin` lesson from the beginning of the course.

**TL;DR:** Be sure the correct kernel pointing to your `uv` virtual environment is selected.

In [ ]:
# ============================================================
# Google Colab Setup — runs only when executed in Colab
# ============================================================
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import importlib
    import os
    import site
    import subprocess

    # Install the course package (published from pyproject.toml) and its pinned extras
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "agentic-ai-engineering-course==0.4.8",
            "nest-asyncio2",
            "google-auth==2.53.0",
            "opentelemetry-api==1.42.1",
            "opentelemetry-sdk==1.42.1",
            "opentelemetry-exporter-otlp-proto-http==1.42.1",
            "opentelemetry-exporter-otlp-proto-common==1.42.1",
            "opentelemetry-proto==1.42.1",
            "jedi==0.18.2",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without restart

    # Load API key from Colab Secrets
    # In Colab: Secrets tab (key icon) → Add new secret → Name: GOOGLE_API_KEY
    from google.colab import userdata

    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    from utils import env

    env.load(required_env_vars=["GOOGLE_API_KEY"])

### Import Key Packages

In [ ]:
import json
from typing import Any

from google import genai
from google.genai import types
from pydantic import BaseModel, Field
from utils import pretty_print

### Initialize the Gemini Client

In [ ]:
client = genai.Client()

### Define Constants

We will use the `gemini-3.5-flash` model, which is fast and cost-effective. We also define a sample financial document that will be used throughout our examples.

In [ ]:
MODEL_ID = "gemini-3.5-flash"

DOCUMENT = """
# Q3 2023 Financial Performance Analysis

The Q3 earnings report shows a 20% increase in revenue and a 15% growth in user engagement, 
beating market expectations. These impressive results reflect our successful product strategy 
and strong market positioning.

Our core business segments demonstrated remarkable resilience, with digital services leading 
the growth at 25% year-over-year. The expansion into new markets has proven particularly 
successful, contributing to 30% of the total revenue increase.

Customer acquisition costs decreased by 10% while retention rates improved to 92%, 
marking our best performance to date. These metrics, combined with our healthy cash flow 
position, provide a strong foundation for continued growth into Q4 and beyond.
"""

## 2. Implementing tool calls from scratch

LLMs are trained on text and can't perform actions in the real world on their own. Tools (or function calling) are the mechanism we use to bridge this gap. We provide the LLM with a list of available tools, and it can decide which one to use and with what arguments to fulfill a user's request.

The process of calling a tool looks as follows:

1. **You:** Send the LLM a prompt and a list of available tools.
2. **LLM:** Responds with a function call request, specifying the tool and arguments.
3. **You:** Execute the requested function in your code.
4. **You:** Send the function's output back to the LLM.
5. **LLM:** Uses the tool's output to generate a final, user-facing response.


### Define Mock Tools

Let's create three simple, mocked functions. One simulates searching Google Drive, another simulates sending a Discord message, and the last one simulates summarizing a document. 

The function signature (input parameters and output type) and docstrings are crucial, as the LLM uses them to understand what each tool does.

In [ ]:
def search_google_drive(query: str) -> dict:
    """
    Searches for a file on Google Drive and returns its content or a summary.

    Args:
        query (str): The search query to find the file, e.g., 'Q3 earnings report'.

    Returns:
        dict: A dictionary representing the search results, including file names and summaries.
    """

    # Here, we mock the response for demonstration.
    # In a real scenario, this would interact with the Google Drive API.
    return {
        "files": [
            {
                "name": "Q3_Earnings_Report_2024.pdf",
                "id": "file12345",
                "content": DOCUMENT,
            }
        ]
    }


def send_discord_message(channel_id: str, message: str) -> dict:
    """
    Sends a message to a specific Discord channel.

    Args:
        channel_id (str): The ID of the channel to send the message to, e.g., '#finance'.
        message (str): The content of the message to send.

    Returns:
        dict: A dictionary confirming the action, e.g., {"status": "success"}.
    """

    # Mocking a successful API call to Discord.
    return {
        "status": "success",
        "status_code": 200,
        "channel": channel_id,
        "message_preview": f"{message[:50]}...",
    }


def summarize_financial_report(text: str) -> str:
    """
    Summarizes a financial report.

    Args:
        text (str): The text to summarize.

    Returns:
        str: The summary of the text.
    """

    # Mocked summary for demonstration.
    return "The Q3 2023 earnings report shows strong performance across all metrics \
with 20% revenue growth, 15% user engagement increase, 25% digital services growth, and \
improved retention rates of 92%."

Now, we need to define the metadata for each function, which will be used as input to the LLM to understand which tool to use and how to call it:

### Exercise 1: Describe your tools to the LLM

The model never sees your Python code. It only sees the schema: name, description, parameters. Writing these by hand once makes clear exactly what frameworks automate later.

**Learning goal:** Author the JSON schema metadata that lets an LLM understand a tool.

**What you need to implement:**

1. Complete the schema for `send_discord_message`: a description of the tool, both parameters (`channel_id` and `message`) with type and description, and the required list
2. Complete the schema for `summarize_financial_report` the same way for its single `text` parameter

**Key concepts:**

- Each parameter entry carries a `type` and a `description` the model reads to fill arguments correctly
- The `required` list tells the model which arguments it cannot omit

**Expected output:** the smoke test prints one line per schema, e.g. `send_discord_message: 2 parameter(s) described, required=['channel_id', 'message']`.

**Implementation hints:**

- The completed `search_google_drive_schema` above your gap is the pattern to follow, and the function docstrings from the previous cell contain the wording you need
- The demo calls further down run either way, but the model only starts picking these tools sensibly once their schemas are complete

In [ ]:
# === Exercise cell: fill in the gaps below ===

search_google_drive_schema = {
    "name": "search_google_drive",
    "description": "Searches for a file on Google Drive and returns its content or a summary.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find the file, e.g., 'Q3 earnings report'.",
            }
        },
        "required": ["query"],
    },
}

# TODO 1: complete the schema for send_discord_message
# (describe the tool, its channel_id and message parameters, and which ones are required)
send_discord_message_schema = {
    "name": "send_discord_message",
    "description": "",
    "parameters": {
        "type": "object",
        "properties": {
            # Your implementation goes here
        },
        "required": [],
    },
}

# TODO 2: complete the schema for summarize_financial_report
# (describe the tool and its text parameter)
summarize_financial_report_schema = {
    "name": "summarize_financial_report",
    "description": "",
    "parameters": {
        "type": "object",
        "properties": {
            # Your implementation goes here
        },
        "required": [],
    },
}

# Smoke test (no API call)
for _schema in (send_discord_message_schema, summarize_financial_report_schema):
    _params = _schema["parameters"]
    print(f"{_schema['name']}: {len(_params['properties'])} parameter(s) described, required={_params['required']}")

### Validation check - run this after your implementation

Uncomment the cell below and run it once Exercise 1 is done. It checks both schemas locally, no API calls.

In [ ]:
# # Validation: check your Exercise 1 implementation
# try:
#     _discord = send_discord_message_schema["parameters"]
#     _summarize = summarize_financial_report_schema["parameters"]
#     assert send_discord_message_schema["description"], "❌ send_discord_message needs a description."
#     assert set(_discord["properties"]) == {"channel_id", "message"}, "❌ send_discord_message should describe exactly channel_id and message."
#     assert set(_discord["required"]) == {"channel_id", "message"}, "❌ Both send_discord_message parameters are required."
#     assert set(_summarize["properties"]) == {"text"}, "❌ summarize_financial_report should describe exactly the text parameter."
#     assert _summarize["required"] == ["text"], "❌ The text parameter is required."
#     assert all(p.get("type") and p.get("description") for p in _discord["properties"].values()), "❌ Every parameter needs a type and a description."
#     print("✅ All checks passed! Your tool schemas are complete.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: mirror the structure of search_google_drive_schema and reuse the docstring wording.")

Ultimately, we will aggregate all the tools in a single dictionary, known as the tools registry:

In [ ]:
TOOLS = {
    "search_google_drive": {
        "handler": search_google_drive,
        "schema": search_google_drive_schema,
    },
    "send_discord_message": {
        "handler": send_discord_message,
        "schema": send_discord_message_schema,
    },
    "summarize_financial_report": {
        "handler": summarize_financial_report,
        "schema": summarize_financial_report_schema,
    },
}
TOOLS_BY_NAME = {tool_name: tool["handler"] for tool_name, tool in TOOLS.items()}
TOOLS_SCHEMA = [tool["schema"] for tool in TOOLS.values()]

Let's take a look at them:

In [ ]:
for tool_name, tool in TOOLS_BY_NAME.items():
    print(f"Tool name: {tool_name}")
    print(f"Tool handler: {tool}")
    print("-" * 75)

In [ ]:
pretty_print.wrapped(json.dumps(TOOLS_SCHEMA[0], indent=2), title="`search_google_drive` Tool Schema")

In [ ]:
pretty_print.wrapped(json.dumps(TOOLS_SCHEMA[1], indent=2), title="`send_discord_message` Tool Schema")

Now, let's see how to call these tools using an LLM. First, we need to define the system prompt:

In [ ]:
TOOL_CALLING_SYSTEM_PROMPT = """
You are a helpful AI assistant with access to tools that enable you to take actions and retrieve information to better 
assist users.

## Tool Usage Guidelines

**When to use tools:**
- When you need information that is not in your training data
- When you need to perform actions in external systems and environments
- When you need real-time, dynamic, or user-specific data
- When computational operations are required

**Tool selection:**
- Choose the most appropriate tool based on the user's specific request
- If multiple tools could work, select the one that most directly addresses the need
- Consider the order of operations for multi-step tasks

**Parameter requirements:**
- Provide all required parameters with accurate values
- Use the parameter descriptions to understand expected formats and constraints
- Ensure data types match the tool's requirements (strings, numbers, booleans, arrays)

## Tool Call Format

When you need to use a tool, output ONLY the tool call in this exact format:

```tool_call
{{"name": "tool_name", "args": {{"param1": "value1", "param2": "value2"}}}}
```

**Critical formatting rules:**
- Use double quotes for all JSON strings
- Ensure the JSON is valid and properly escaped
- Include ALL required parameters
- Use correct data types as specified in the tool definition
- Do not include any additional text or explanation in the tool call

## Response Behavior

- If no tools are needed, respond directly to the user with helpful information
- If tools are needed, make the tool call first, then provide context about what you're doing
- After receiving tool results, provide a clear, user-friendly explanation of the outcome
- If a tool call fails, explain the issue and suggest alternatives when possible

## Available Tools

<tool_definitions>
{tools}
</tool_definitions>

Your goal is to be maximally helpful to the user. Use tools when they add value, but don't use them unnecessarily.
"""

Let's try the prompt with a few examples.

In [ ]:
USER_PROMPT = """
Can you help me find the latest quarterly report and share key insights with the team?
"""

messages = [TOOL_CALLING_SYSTEM_PROMPT.format(tools=str(TOOLS_SCHEMA)), USER_PROMPT]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
)

pretty_print.wrapped(response.text, title="LLM Tool Call Response")

In [ ]:
USER_PROMPT = """
Send a greeting message to the #finance channel on Discord.
"""

messages = [TOOL_CALLING_SYSTEM_PROMPT.format(tools=str(TOOLS_SCHEMA)), USER_PROMPT]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
)
pretty_print.wrapped(response.text, title="LLM Tool Call Response")

The next step is to parse the LLM response and call the tool using Python.

First, we parse the LLM output to extract the JSON from the response:

### Exercise 2: Extract the tool call from raw text

With the from-scratch approach, the tool call arrives as plain text inside a ```` ```tool_call ```` fence. Before anything can execute, that block has to be isolated.

**Learning goal:** Parse a fenced tool call block out of a raw LLM response.

**What you need to implement:**

1. Isolate whatever comes after the opening ```` ```tool_call ```` fence
2. Cut it off at the closing ```` ``` ```` fence
3. Return the trimmed result

**Key concepts:**

- `str.split(separator)` breaks a string into pieces around a separator, indexing picks the piece you want
- `str.strip()` trims surrounding whitespace and newlines

**Expected output:** the inline smoke test prints `{"name": "demo_tool", "args": {"x": 1}}`.

**Implementation hints:**

- Two splits are enough, one per fence
- The `json.loads` cell just below consumes `tool_call_str` from the real response, it will error until this function returns real JSON

In [ ]:
# === Exercise cell: fill in the gaps below ===


def extract_tool_call(response_text: str) -> str:
    """
    Extracts the tool call from the response text.

    Steps to complete:
    1. Isolate the text after the opening ```tool_call fence
    2. Cut it off at the closing ``` fence
    3. Return the trimmed string
    """
    # Your implementation goes here

    return ""  # Replace with the extracted tool call string


# Smoke test (no API call): should print {"name": "demo_tool", "args": {"x": 1}}
_sample = 'Sure, calling it now:\n```tool_call\n{"name": "demo_tool", "args": {"x": 1}}\n```\nDone.'
print(extract_tool_call(_sample) or "(empty, complete the function above)")

# Extract from the real LLM response generated above
tool_call_str = extract_tool_call(response.text)
tool_call_str

Next, we parse the stringified JSON to a Python dict:

In [ ]:
tool_call = json.loads(tool_call_str)
tool_call

Now, we retrieve the tool handler, which is a Python function:

In [ ]:
tool_handler = TOOLS_BY_NAME[tool_call["name"]]
tool_handler

Ultimately, we call the Python function using the arguments generated by the LLM:

In [ ]:
tool_result = tool_handler(**tool_call["args"])
pretty_print.wrapped(tool_result, indent=2, title="LLM Tool Call Response")

We can summarize the tool execution in the following function:

In [ ]:
def call_tool(response_text: str, tools_by_name: dict) -> Any:
    """
    Call a tool based on the response from the LLM.

    Args:
        response_text (str): The raw response text from the LLM containing the tool call.
        tools_by_name (dict): Dictionary mapping tool names to their handler functions.

    Returns:
        Any: The result of executing the tool with the provided arguments.
    """

    tool_call_str = extract_tool_call(response_text)
    tool_call = json.loads(tool_call_str)
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]
    tool = tools_by_name[tool_name]

    return tool(**tool_args)

In [ ]:
pretty_print.wrapped(json.dumps(call_tool(response.text, tools_by_name=TOOLS_BY_NAME), indent=2), title="LLM Tool Call Response")

Usually, before showing it to the user, we want the LLM to interpret the tool output:

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents=f"Interpret the tool result: {json.dumps(tool_result, indent=2)}",
)
pretty_print.wrapped(response.text, title="LLM Tool Call Response")

That's the basic concept of tool calling! We've successfully implemented function calling from scratch.

## 3. Implementing a tool calling framework from scratch

For a better analogy with what we see in frameworks such as LangGraph or MCP, let's define a `@tool` decorator that automatically computes the schemas defined above based on the function signature and docstring.

First, we will define the `ToolFunction` class that aggregates the function's schema:

In [ ]:
from inspect import Parameter, signature
from typing import Any, Callable, Dict


class ToolFunction:
    def __init__(self, func: Callable, schema: Dict[str, Any]) -> None:
        self.func = func
        self.schema = schema
        self.__name__ = func.__name__
        self.__doc__ = func.__doc__

    def __call__(self, *args: Any, **kwargs: Any) -> Any:
        return self.func(*args, **kwargs)

Now, let's define a `tools` registry that will aggregate all our decorated tools:

In [ ]:
tools: list[ToolFunction] = []

Ultimately, let's define the actual `@tool` decorator:

### Exercise 3: Build the @tool decorator

This is the heart of the from-scratch framework: a decorator that inspects any Python function and generates its schema automatically, exactly what LangGraph-style frameworks do for you.

**Learning goal:** Generate a tool schema from a function's signature and docstring using introspection.

**What you need to implement:**

1. Inspect the function's signature
2. Build a properties dict with one entry per parameter (skip `self`), defaulting the type to string with a simple generated description
3. Collect every parameter without a default value into a required list
4. Assemble the schema: the function's name, its docstring as the description, and the parameters object shaped like the hand-written schemas from Exercise 1
5. Wrap function and schema in a `ToolFunction`, append it to the `tools` registry, and return it

**Key concepts:**

- `signature(func)` from `inspect` exposes `.parameters`, an ordered mapping of name to `Parameter`
- `param.default == Parameter.empty` is how you detect a parameter with no default
- `func.__name__` and `func.__doc__` supply the metadata

**Expected output:** after running the decorated examples in the next cell, the registry inspection cells show three `ToolFunction` entries, and `tools[0].schema["name"]` is `search_google_drive_example`.

**Implementation hints:**

- The schema you assemble here has exactly the same shape you hand-wrote in Exercise 1, that was the dress rehearsal
- The placeholder returns an unregistered wrapper, so the registry inspection cells below stay empty (and `tools[0]` raises an IndexError) until you implement and re-run the decorated functions

In [ ]:
# === Exercise cell: fill in the gaps below ===


def tool() -> Callable[[Callable], ToolFunction]:
    """
    A decorator that creates a tool schema from a function.

    Returns:
        A decorator function that wraps the original function and adds a schema
    """

    def decorator(func: Callable) -> ToolFunction:
        """
        Steps to complete:
        1. Inspect the function's signature
        2. Build a properties dict with one entry per parameter (skip 'self'),
           using a string type and a simple description as defaults
        3. Add every parameter without a default value to a required list
        4. Assemble the schema dict (name, description from the docstring,
           and the parameters object)
        5. Wrap the function and schema in a ToolFunction, append it to the
           `tools` registry, and return it
        """
        # Your implementation goes here

        return ToolFunction(func, {})  # Replace: build the real schema and register the tool

    return decorator

Let's redefine our tools leveraging the `@tool` decorator:

In [ ]:
@tool()
def search_google_drive_example(query: str) -> dict:
    """Search for files in Google Drive."""
    return {"files": ["Q3 earnings report"]}


@tool()
def send_discord_message_example(channel_id: str, message: str) -> dict:
    """Send a message to a Discord channel."""
    return {"message": "Message sent successfully"}


@tool()
def summarize_financial_report_example(text: str) -> str:
    """Summarize the contents of a financial report."""
    return "Financial report summarized successfully"

### Validation check - run this after your implementation

Uncomment the cell below and run it once you have implemented the decorator and run the cell above, which decorates the three example functions and fills the `tools` registry. If you change the decorator, re-run that cell before validating. No API calls.


In [ ]:
# # Validation: check your Exercise 3 implementation
# try:
#     assert tools, "❌ The tools registry is empty. Implement the decorator, then re-run the cell defining the example tools."
#     _by_name = {t.schema.get("name"): t for t in tools if isinstance(t, ToolFunction)}
#     assert "send_discord_message_example" in _by_name, "❌ send_discord_message_example is not registered with a named schema."
#     _discord = _by_name["send_discord_message_example"].schema["parameters"]
#     assert set(_discord["properties"]) == {"channel_id", "message"}, "❌ Expected properties for channel_id and message."
#     assert set(_discord["required"]) == {"channel_id", "message"}, "❌ Both parameters lack defaults, so both are required."
#     assert _by_name["send_discord_message_example"].schema["description"], "❌ The schema description should come from the docstring."
#     print("✅ All checks passed! Your decorator generates schemas automatically.")
# except AssertionError as e:
#     print(e)
#     print("💡 Tip: loop over signature(func).parameters.items() and mirror the schema shape from Exercise 1.")

Let's inspect the `tools` registry to look at all the available tools:

In [ ]:
tools

The first tool from the registry:

In [ ]:
tools[0].schema["name"]

We can see that the first tool from the registry is `search_google_drive_example`. As expected, after the function has been decorated, it has been wrapped into a `ToolFunction` object:

In [ ]:
type(tools[0])

It has automatically computed the tool schema that will be passed to the LLM:

In [ ]:
pretty_print.wrapped(json.dumps(tools[0].schema, indent=2), title="Search Google Drive Example")

...and contains the actual function handler:

In [ ]:
search_google_drive_example.func

Let's see how this new method works with LLMs. First, we have to create our tool mappings:

In [ ]:
tools_by_name = {tool.schema["name"]: tool.func for tool in tools}
tools_schema = [tool.schema for tool in tools]

In [ ]:
pretty_print.wrapped(json.dumps(tools_schema, indent=2), title="Tools Schema")

Now, let's call the LLM by passing the tool schemas, as before:

In [ ]:
USER_PROMPT = """
Can you help me find the latest quarterly report and share key insights with the team?
"""

messages = [TOOL_CALLING_SYSTEM_PROMPT.format(tools=str(tools_schema)), USER_PROMPT]

response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
)
pretty_print.wrapped(response.text, title="LLM Tool Call Response")

In [ ]:
pretty_print.wrapped(json.dumps(call_tool(response.text, tools_by_name=tools_by_name), indent=2), title="LLM Tool Call Response")

Voilà! We have our little tool calling framework.

## 4. Implementing production-level tool calls with Gemini

In production, most of the time, we don't implement tool calling from scratch. Instead, we leverage the native interface of a specific API such as Gemini or OpenAI. So, let's see how we can use Gemini's built-in tool calling capabilities instead of our custom implementation.

### Exercise 4: Switch to Gemini's native tool calling

No more system prompt full of formatting rules and no more text parsing: the provider accepts your schemas directly through the request config and returns typed function call objects.

**Learning goal:** Configure native function calling with the Gemini API.

**What you need to implement:**

1. Wrap the hand-written `search_google_drive_schema` and `send_discord_message_schema` as native function declarations inside a `types.Tool`, assigned to `tools`
2. Build a `types.GenerateContentConfig` that carries those tools plus a tool config whose function calling mode forces the model to always predict a function call, assigned to `config`

**Key concepts:**

- `types.FunctionDeclaration` accepts the same name/description/parameters fields you wrote by hand, dict unpacking maps them directly
- `types.ToolConfig` wraps a `types.FunctionCallingConfig`, whose mode controls whether function calls are optional, forbidden, or mandatory

**Expected output:** the smoke test prints the two declared function names, and the demo cell below then shows a typed function call object instead of raw text.

**Implementation hints:**

- The demo cell below still runs while `config` is `None`, but as a plain text call whose function call prints as `None`, that is your signal this exercise is not done yet

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Build the native tools list: one types.Tool wrapping function declarations
#    created from search_google_drive_schema and send_discord_message_schema
# 2. Build the generation config: pass the tools and a tool_config that forces
#    the model to always predict a function call

tools = None  # TODO 1: the native tools list
config = None  # TODO 2: the generation config

# Your implementation goes here


# Smoke test (no API call)
if tools is not None and config is not None:
    print("Declared functions:", [fd.name for t in tools for fd in t.function_declarations])
else:
    print("(tools/config not set yet, complete the TODOs above)")

As you can see, when calling the LLM, we don't have to explicitly define a system prompt that guides the LLM on how to use the tools. Instead, we pass the tool schema to the LLM provider through the config, which will handle tool calling internally. This is more efficient, as they take care of optimizing tool/function calling for each specific model:

In [ ]:
pretty_print.wrapped(USER_PROMPT, title="User Prompt")
response = client.models.generate_content(
    model=MODEL_ID,
    contents=USER_PROMPT,
    config=config,
)
pretty_print.wrapped(str(response.candidates[0].content.parts[0].function_call), title="LLM Response - Function Call")

To simplify the implementation even more, Google's genai supports taking Python functions directly as input. Now, the SDK creates the schema based on the signature, type hints and pydocs:

In [ ]:
client = genai.Client()
config = types.GenerateContentConfig(
    tools=[search_google_drive, send_discord_message],
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="ANY")),
)

Now, let's call the LLM again using the new config:

In [ ]:
pretty_print.wrapped(USER_PROMPT, title="User Prompt")
response = client.models.generate_content(
    model=MODEL_ID,
    contents=USER_PROMPT,
    config=config,
)
pretty_print.wrapped(str(response.candidates[0].content.parts[0].function_call), title="LLM Response - Function Call")

Let's look at the LLM response better:

In [ ]:
response_message_part = response.candidates[0].content.parts[0]
function_call = response_message_part.function_call
function_call

In [ ]:
pretty_print.wrapped(function_call.args, title="Function Call Args")

In [ ]:
tool_handler = TOOLS_BY_NAME[function_call.name]
tool_handler

In [ ]:
tool_handler(**function_call.args)

Now let's create a simplified function that works with Gemini's native function call objects:

In [ ]:
def call_tool(function_call) -> Any:
    tool_name = function_call.name
    tool_args = function_call.args

    tool_handler = TOOLS_BY_NAME[tool_name]

    return tool_handler(**tool_args)

In [ ]:
tool_result = call_tool(response_message_part.function_call)
pretty_print.wrapped(tool_result, indent=2, title="Tool Result")

## 5. Using Pydantic models as tools for on-demand structured outputs

When it comes to structured outputs, a more elegant and powerful pattern is to treat our Pydantic model *as a tool*. We can ask the model to "call" this Pydantic tool, and the arguments it generates will be our structured data.

This combines the power of function calling with the robustness of Pydantic for structured data extraction. It's the recommended approach for complex data extraction tasks.

Let's define the same Pydantic model as in the structured outputs lesson:

In [ ]:
class DocumentMetadata(BaseModel):
    """Pydantic class to hold structured metadata for a document."""

    summary: str = Field(description="A concise, 1-2 sentence summary of the document.")
    tags: list[str] = Field(description="A list of 3-5 high-level tags relevant to the document.")
    keywords: list[str] = Field(description="A list of specific keywords or concepts mentioned.")
    quarter: str = Field(description="The quarter of the financial year described in the document (e.g., Q3 2023).")
    growth_rate: str = Field(description="The growth rate of the company described in the document (e.g., 10%).")

Now, let's see how to use it as a tool:

### Exercise 5: Declare a Pydantic model as a tool

Structured extraction and function calling meet here: the model "calls" a function whose parameters are your Pydantic schema, and the generated arguments are your structured data.

**Learning goal:** Turn a Pydantic model into a callable tool declaration.

**What you need to implement:**

1. Create a function declaration named `extract_metadata` with a short description of what it extracts
2. Use the JSON Schema generated from `DocumentMetadata` as the declaration's parameters
3. Wrap the declaration in a `types.Tool` assigned to `extraction_tool`

**Key concepts:**

- `DocumentMetadata.model_json_schema()` produces the parameters schema, the same method you used in Lesson 4
- The declaration's name is what shows up as the "called" function in the response

**Expected output:** the smoke test prints `Declared tool: extract_metadata`, and the call cells below then produce a function call whose args validate cleanly into `DocumentMetadata`.

**Implementation hints:**

- The config and call cells below this one will error while `extraction_tool` is `None`, they come alive once this is implemented

In [ ]:
# === Exercise cell: fill in the gaps below ===

# The Pydantic class 'DocumentMetadata' is now our 'tool'
# Steps to complete:
# 1. Create a function declaration named "extract_metadata" with a short description
# 2. Use the JSON Schema generated from DocumentMetadata as its parameters
# 3. Wrap it in a types.Tool and assign it to extraction_tool

extraction_tool = None  # TODO: build the extraction tool

# Your implementation goes here


# Smoke test (no API call)
if extraction_tool is not None:
    print("Declared tool:", extraction_tool.function_declarations[0].name)
else:
    print("(extraction_tool not set yet, complete the TODO above)")

Ultimately, we define the config:

In [ ]:
config = types.GenerateContentConfig(
    tools=[extraction_tool],
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="ANY")),
)

Now we call the LLM:

In [ ]:
prompt = f"""
Please analyze the following document and extract its metadata.

Document:
<document>
{DOCUMENT}
</document>
"""

response = client.models.generate_content(model=MODEL_ID, contents=prompt, config=config)
response_message_part = response.candidates[0].content.parts[0]

Print the output:

In [ ]:
function_call = response_message_part.function_call
pretty_print.function_call(function_call, title="Function Call")

Let's validate the output using Pydantic:

In [ ]:
try:
    document_metadata = DocumentMetadata(**function_call.args)
    pretty_print.wrapped("Validation successful!")
except Exception as e:
    pretty_print.wrapped(str(e), title="Validation Error")

## 6. The downsides of running tools in a loop

Now, let's implement a more sophisticated approach where we put tool calling in a loop with a conversation history. This allows the agent to perform multi-step tasks by calling multiple tools in sequence. Let's create a scenario where we ask the agent to find a report on Google Drive and then communicate its findings on Discord.

First, we define the config:

In [ ]:
tools = [
    types.Tool(
        function_declarations=[
            types.FunctionDeclaration(**search_google_drive_schema),
            types.FunctionDeclaration(**send_discord_message_schema),
            types.FunctionDeclaration(**summarize_financial_report_schema),
        ]
    )
]
config = types.GenerateContentConfig(
    tools=tools,
    tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(mode="ANY")),
)

Next, the user prompt:

In [ ]:
USER_PROMPT = """
Please find the Q3 earnings report on Google Drive and send a summary of it to 
the #finance channel on Discord.
"""

Now, we make the first LLM call as always:

In [ ]:
messages = [USER_PROMPT]

pretty_print.wrapped(USER_PROMPT, title="User Prompt")
response = client.models.generate_content(
    model=MODEL_ID,
    contents=messages,
    config=config,
)
response_message_part = response.candidates[0].content.parts[0]
pretty_print.function_call(response_message_part.function_call, title="Function Call")

messages.append(response.candidates[0].content)

Ultimately, we add the LLM in a loop until it doesn't return new `function_call` objects or it hits the `max_iterations` limit:

### Exercise 6: Run tools in a loop

One tool call rarely finishes a multi-step task. The naive approach chains calls: execute the requested tool, feed the result back, let the model request the next one, until it stops or hits an iteration cap.

**Learning goal:** Implement the execute-feed-back-repeat loop that lets a model chain tool calls.

**What you need to implement:**

1. Loop while the response part carries a function call and iterations remain
2. Execute the requested tool with `call_tool` and print the result
3. Package the tool result as a function response part and append it to `messages`
4. Call the model again with the accumulated messages and the same config, refresh `response_message_part`, append the new content to `messages`, and decrement the counter

**Key concepts:**

- `types.Part.from_function_response(name=..., response=...)` wraps a tool result so the model recognizes it as the answer to its call, pass the result inside a dict
- The `messages` list alternates: user prompt, tool call, tool result, tool call, tool result...
- `pretty_print.function_call(part.function_call, only_name=True, ...)` prints just the name for compact loop output

**Expected output:** for the Drive-then-Discord request: a search tool result, a summarize call, a Discord send call with a status confirmation, then the final agent response block.

**Implementation hints:**

- Everything you need already exists: `call_tool`, `messages`, `config`, and `response_message_part` from the first call above
- Until you implement the loop, the cell just reprints the first function call as the final response, no extra API calls happen

In [ ]:
# === Exercise cell: fill in the gaps below ===

# Steps to complete:
# 1. Loop while response_message_part has a function call and max_iterations > 0
# 2. Execute the requested tool via call_tool and pretty-print the result
# 3. Wrap the result with types.Part.from_function_response and append it to messages
# 4. Call the model again with messages + config, refresh response_message_part,
#    append the new response content to messages, and decrement max_iterations

max_iterations = 3

# Your implementation goes here


pretty_print.function_call(response.candidates[0].content.parts[0].function_call, title="Final Agent Response")

Running tools in a loop is powerful for multi-step tasks, but this naive approach has limitations. 

It doesn't provide explicit opportunities for the model to reason about tool outputs before deciding on the next action. The agent immediately moves to the next function call without pausing to think about what it learned or whether it should change strategy.

This limitation leads us to more sophisticated patterns like **ReAct** (Reasoning and Acting), which explicitly interleaves reasoning steps with tool calls, allowing the agent to think through problems more deliberately. We will explore ReAct patterns in lessons 7 and 8.

## Stretch challenges

Want to go further? Try these on your own:

1. Extend the `@tool` decorator to read real parameter types from type hints (`str`, `int`, `bool`) instead of defaulting everything to string.
2. Add a `get_current_date` tool to the loop scenario and ask the agent to include the date in its Discord message, watch how the call order changes.
3. The loop's `while` condition checks `hasattr`, but a Part object always has a `function_call` attribute (it may just be `None`). Rewrite the condition to check the value and switch the function calling mode from forced to automatic, then see how the loop terminates naturally with a text answer.